# ISFEST 2026 DATA COMPETITION (UNIVERSITAS MULTIMEDIA NUSANTARA)
## Studi Kasus: Mapping the Demand for Electric Vehicle (EV) Charging Infrastructure
### Kerangka Kerja: Cross-Industry Standard Process for Data Mining (CRISP-DM)
### Arsitektur: Calibrated Multi-Seed Tri-Model Stacking & Macro-Spatial Bayesian Profiles (Run 6 Terkalibrasi - Target 0.067xx)

---
**Identitas Tim:**
* **Nama Tim:** MAKAN ITU PENTING
* **Anggota Tim:**
  1. [Nama Anggota 1] (Ketua Tim)
  2. [Nama Anggota 2]
  3. [Nama Anggota 3]
* **Institusi:** Universitas Negeri Surabaya (UNESA)
* **Deskripsi Singkat:** Notebook ini merepresentasikan solusi analitis mandiri resmi (Run 6 Terkalibrasi) berbasis metodologi standar industri CRISP-DM 6 fase untuk memprediksi tingkat pemanfaatan stasiun pengisian kendaraan listrik (EV charging utilization rate). Arsitektur pemodelan menggabungkan 5 pilar profil target makro Bayesian m-estimate (m=15), indikator deviasi cuaca lokal kota (temp_dev_city_hour), disparitas ekonomi bahan bakar (gas_price_ratio_city), serta interaksi klaster jam sibuk diurnal. Tiga model gradient boosting heterogen terkuat (LightGBM leaf-wise, CatBoost GPU oblivious trees, dan XGBoost GPU histogram) dilatih dengan mekanisme early stopping presisi (40 rounds) untuk mencegah overfitting, kemudian di-fit pada seluruh data menggunakan teknik peredaman variansi multi-seed (Seeds 42, 100, 2024). Penggabungan akhir dilakukan menggunakan Non-Negative Ridge Stacking Meta-Learner teratur, batas fisik operasional [0.02, 0.98], dan pembulatan kuantisasi sensor 3 desimal. Seluruh proses berjalan secara terpadu di dalam satu notebook tunggal tanpa dependensi eksternal untuk mendukung pencapaian **SDG 7: Affordable and Clean Energy** dan **SDG 9: Industry, Innovation, and Infrastructure**.
---

# 1. Business Understanding (CRISP-DM Fase 1)

### 1.1 Latar Belakang dan Konteks Industri
Akselerasi adopsi kendaraan listrik (Electric Vehicle / EV) sebagai pilar dekarbonisasi transportasi menghadapi tantangan kritis berupa ketimpangan pemanfaatan fasilitas pengisian daya (utilization disparity). Di simpul transportasi perkotaan dan koridor jalan tol, antrean kendaraan pada jam sibuk kerap menimbulkan ketidaknyamanan pengemudi (range anxiety) serta membebani transformator gardu listrik lokal. Sebaliknya, stasiun di kawasan pemukiman seringkali mengalami underutilization, yang menyebabkan inefisiensi pengembalian modal investasi bagi operator (Charge Point Operators / CPO).

### 1.2 Rumusan Masalah dan Keselarasan terhadap Sasaran Pembangunan Berkelanjutan (SDG)
Ketidakseimbangan beban dipicu oleh interaksi kompleks antara pola temporal harian dan mingguan, spesifikasi teknis kelistrikan (daya keluaran port dan total kapasitas), faktor termodinamika cuaca (suhu lingkungan dan presipitasi), serta amenitas di sekitar fasilitas. Prediksi akurat terhadap pemanfaatan stasiun secara granular sangat krusial untuk:
1. **SDG 7 (Affordable and Clean Energy)**: Optimalisasi manajemen beban puncak (peak shaving), pencegahan kelebihan beban transformator lokal, dan efisiensi konsumsi energi bersih.
2. **SDG 9 (Industry, Innovation, and Infrastructure)**: Memberikan acuan analitis berbasis bukti dalam perencanaan ekspansi port dan penempatan modul pengisi daya cepat (DC Fast Chargers).
3. **SDG 11 (Sustainable Cities and Communities)**: Mereduksi emisi perkotaan dan waktu antrean pengisian daya di koridor mobilitas publik.

### 1.3 Tujuan Pemodelan dan Spesifikasi Metrik Evaluasi
Tujuan analitis adalah memprediksi nilai kontinu `utilization_rate` (rentang [0.0, 1.0]) pada setiap stasiun per interval 30 menit. Kinerja dievaluasi secara resmi menggunakan metrik **Root Mean Squared Error (RMSE)**:

RMSE = sqrt( (1 / N) * sum( (y_i - y_hat_i)^2 ) )

di mana N adalah total baris data pengujian, y_i adalah nilai aktual, dan y_hat_i adalah nilai estimasi model prediktif.

# 2. Data Understanding (CRISP-DM Fase 2)

Tahap ini mencakup inisialisasi lingkungan komputasi, deteksi perangkat keras GPU Tesla P100, pemuatan data efisien, audit tiga anomali spesifik yang diidentifikasi oleh dewan juri, serta eksplorasi karakteristik sebaran data.

In [21]:
# Konfigurasi Pustaka dan Pengaturan Lingkungan Komputasi
import os
import gc
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

warnings.filterwarnings('ignore')

# Standar Visualisasi Profesional
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11

OUTPUT_FIG_DIR = './figures'
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)

# Deteksi Akselerasi Perangkat Keras GPU Tesla P100
gpu_available = False
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_available = True
        print(f"[INFO] Akselerator GPU Terdeteksi: {gpu_name} (Siap untuk CatBoost & XGBoost)")
    else:
        print("[INFO] Akselerator GPU tidak aktif. Berjalan pada Multi-Threaded CPU.")
except ImportError:
    print("[INFO] Lingkungan standar terdeteksi.")


[INFO] Akselerator GPU Terdeteksi: Tesla P100-PCIE-16GB (Siap untuk CatBoost & XGBoost)


In [22]:
# Pemuatan Data Pelatihan dan Pengujian dengan Optimasi Memori
CANDIDATE_DIRS = [
    '/kaggle/input/datasets/rabbaniyuki/isfest-dataset',
    '/kaggle/input/isfest-dataset',
    '/kaggle/input/ev-charging-demand-indonesian-student-competition',
    './data',
    '.'
]

def locate_data_file(filename):
    for directory in CANDIDATE_DIRS:
        full_path = os.path.join(directory, filename)
        if os.path.exists(full_path):
            return full_path
    raise FileNotFoundError(f"Berkas {filename} tidak ditemukan pada jalur kandidat.")

train_path = locate_data_file('train.csv')
test_path = locate_data_file('test.csv')

print(f"[INFO] Membaca data latih dari : {train_path}")
train_df = pd.read_csv(train_path)
print(f"[INFO] Membaca data uji dari   : {test_path}")
test_df = pd.read_csv(test_path)

def optimize_memory(df):
    initial_mem = df.memory_usage().sum() / (1024**2)
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not str(col_type).startswith('datetime'):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type).startswith('int'):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            elif str(col_type).startswith('float'):
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    final_mem = df.memory_usage().sum() / (1024**2)
    print(f"[INFO] Pengurangan memori: {initial_mem:.1f} MB -> {final_mem:.1f} MB (Efisiensi: {100*(initial_mem-final_mem)/initial_mem:.1f}%)")
    return df

train_df = optimize_memory(train_df)
test_df = optimize_memory(test_df)

print(f"[INFO] Dimensi Data Latih: {train_df.shape[0]:,} baris x {train_df.shape[1]} kolom")
print(f"[INFO] Dimensi Data Uji  : {test_df.shape[0]:,} baris x {test_df.shape[1]} kolom")


[INFO] Membaca data latih dari : /kaggle/input/datasets/rabbaniyuki/isfest-dataset/train.csv
[INFO] Membaca data uji dari   : /kaggle/input/datasets/rabbaniyuki/isfest-dataset/test.csv
[INFO] Pengurangan memori: 168.0 MB -> 133.0 MB (Efisiensi: 20.8%)
[INFO] Pengurangan memori: 40.2 MB -> 32.4 MB (Efisiensi: 19.4%)
[INFO] Dimensi Data Latih: 1,048,575 baris x 21 kolom
[INFO] Dimensi Data Uji  : 263,550 baris x 20 kolom


In [23]:
# Audit Tiga Anomali Data Sesuai Panduan Panitia ISFEST 2026
print("=== [AUDIT ANOMALI 1: Stasiun Bernama Sama dengan ID Berbeda] ===")
st_grouped = train_df.groupby('station_name')['station_id'].nunique()
duplicate_stations = st_grouped[st_grouped > 1]
for st_name, count in duplicate_stations.items():
    ids = train_df[train_df['station_name'] == st_name]['station_id'].unique().tolist()
    print(f"Stasiun '{st_name}' memiliki {count} ID berbeda: {ids}")
print("-> Keputusan: station_id ditetapkan sebagai entitas spasial primer pemodelan.\n")

print("=== [AUDIT ANOMALI 2: Pola Nilai Kosong (Missing Values)] ====")
missing_tr = train_df.isnull().sum()[train_df.isnull().sum() > 0]
missing_te = test_df.isnull().sum()[test_df.isnull().sum() > 0]
missing_table = pd.DataFrame({'Train Nulls': missing_tr, 'Test Nulls': missing_te})
print(missing_table)
print("-> Keputusan: Dilakukan imputasi temporal terarah forward/backward fill per stasiun dan median kota.\n")

print("=== [AUDIT ANOMALI 3: Celah Kontinuitas 31 Desember 2025] ====")
test_dt_audit = pd.to_datetime(test_df['timestamp'], format='mixed')
dec31_rows = test_df[test_dt_audit.dt.date == pd.to_datetime('2025-12-31').date()]
print(f"Jumlah pencatatan pada 31 Desember 2025: {len(dec31_rows)} baris.")
print(f"Jam yang tercatat: {test_dt_audit[dec31_rows.index].dt.hour.unique().tolist()} (Hanya jam 00:00).")
print("-> Keputusan: Celah waktu diantisipasi dengan rekayasa fitur berbasis kalender murni.\n")


=== [AUDIT ANOMALI 1: Stasiun Bernama Sama dengan ID Berbeda] ===
Stasiun 'Electrify America - Phoenix #11' memiliki 2 ID berbeda: ['EV00131', 'EV00071']
Stasiun 'Volta - San Diego #10' memiliki 2 ID berbeda: ['EV00010', 'EV00070']
-> Keputusan: station_id ditetapkan sebagai entitas spasial primer pemodelan.

=== [AUDIT ANOMALI 2: Pola Nilai Kosong (Missing Values)] ====
                  Train Nulls  Test Nulls
temperature_f           43905       10928
precipitation_mm        23902        6178
-> Keputusan: Dilakukan imputasi temporal terarah forward/backward fill per stasiun dan median kota.

=== [AUDIT ANOMALI 3: Celah Kontinuitas 31 Desember 2025] ====
Jumlah pencatatan pada 31 Desember 2025: 150 baris.
Jam yang tercatat: [0] (Hanya jam 00:00).
-> Keputusan: Celah waktu diantisipasi dengan rekayasa fitur berbasis kalender murni.



# 3. Data Preparation (CRISP-DM Fase 3)

### 3.1 Peningkatan Rekayasa Fitur Komprehensif (Formula Jawara Run 6)
Untuk mengeliminasi galat jam sibuk dan mereduksi variansi stokastik, direkayasa fitur prediktif domain tingkat lanjut:
1. **Waktu Granular Kontinu**: `time_float = hour + minute / 60.0` (resolusi 48 interval) dan transformasi trigonometri siklikal `sin_hour`, `cos_hour`, `sin_dow`, `cos_dow`.
2. **Termodinamika Baterai EV**: `battery_cold_penalty = max(0, 32 - temp)` untuk memetakan perlambatan laju penerimaan arus baterai lithium-ion pada suhu beku, penanda panas ekstrem (`is_extreme_heat`), serta hujan lebat (`is_raining`).
3. **Anomali Cuaca Relatif terhadap Kota (`temp_dev_city_hour`)**: Mengukur deviasi suhu stasiun terhadap rata-rata temperatur kota pada jam tersebut untuk menangkap fluktuasi iklim mikro lokal.
4. **Disparitas Tekanan Ekonomi Energi**: Rasio harga bensin lokal terhadap rata-rata kota (`gas_price_ratio_city`) dan rasio per unit daya (`gas_price_per_kw`).
5. **5 Pilar Interaksi Klaster Jam Sibuk Diurnal**: Pemetaan beban pengisian pada stasiun perkantoran (`is_workplace_peak`), pusat perbelanjaan (`is_mall_peak`), koridor jalan tol (`is_highway_peak`), pemukiman malam (`is_residential_night`), serta kondisi cuaca beku di jalan tol (`freezing_highway`).
6. **5 Pilar Profil Target Makro Bayesian m-estimate (m=15)**: Agregasi bebas kebocoran pada kombinasi:
   - `target_prof_st_hr_wk` (stasiun x jam x akhir pekan)
   - `target_prof_st_hr` (stasiun x jam)
   - `target_prof_st` (stasiun global)
   - `target_prof_loc_hr` (tipe lokasi x jam)
   - `target_prof_net_hr` (operator jaringan x jam)

In [24]:
# Rekayasa Fitur Spatio-Temporal, Termodinamika, dan Domain Spesifik
train_ext = train_df.copy()
test_ext = test_df.copy()

train_ext['datetime'] = pd.to_datetime(train_ext['timestamp'], format='mixed')
test_ext['datetime'] = pd.to_datetime(test_ext['timestamp'], format='mixed')

for df in [train_ext, test_ext]:
    # 1. Komponen Waktu Granular dan Kalender
    df['hour'] = df['datetime'].dt.hour
    df['minute'] = df['datetime'].dt.minute
    df['time_float'] = (df['hour'] + df['minute'] / 60.0).astype(np.float32)
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['day'] = df['datetime'].dt.day
    df['month'] = df['datetime'].dt.month
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['weekofyear'] = df['datetime'].dt.isocalendar().week.astype(int)
    
    # 2. Transformasi Trigonometri Siklikal
    df['sin_hour'] = np.sin(2 * np.pi * df['time_float'] / 24.0).astype(np.float32)
    df['cos_hour'] = np.cos(2 * np.pi * df['time_float'] / 24.0).astype(np.float32)
    df['sin_dow'] = np.sin(2 * np.pi * df['dayofweek'] / 7.0).astype(np.float32)
    df['cos_dow'] = np.cos(2 * np.pi * df['dayofweek'] / 7.0).astype(np.float32)
    
    # 3. Imputasi Temporal Terarah
    df['temperature_f'] = df.groupby('station_id')['temperature_f'].ffill().bfill()
    df['precipitation_mm'] = df.groupby('station_id')['precipitation_mm'].ffill().bfill()
    df['temperature_f'] = df.groupby('city')['temperature_f'].transform(lambda x: x.fillna(x.median()))
    df['precipitation_mm'] = df.groupby('city')['precipitation_mm'].transform(lambda x: x.fillna(0.0))
    
    # 4. Termodinamika Baterai & Cuaca Ekstrem
    df['is_freezing'] = ((df['temperature_f'] <= 32.0) | (df['weather_condition'] == 'freezing')).astype(int)
    df['battery_cold_penalty'] = np.maximum(0.0, 32.0 - df['temperature_f']).astype(np.float32)
    df['is_extreme_heat'] = ((df['temperature_f'] >= 95.0) | (df['weather_condition'] == 'extreme_heat')).astype(int)
    df['is_raining'] = (df['precipitation_mm'] > 0.0).astype(int)
    
    # 5. Anomali Suhu Relatif terhadap Rata-rata Kota pada Jam Tersebut
    city_hr_temp = df.groupby(['city', 'hour'])['temperature_f'].transform('mean')
    df['temp_dev_city_hour'] = (df['temperature_f'] - city_hr_temp).astype(np.float32)
    
    # 6. Disparitas Tekanan Ekonomi Bahan Bakar Minyak
    city_gas_avg = df.groupby('city')['gas_price_per_gallon'].transform('mean')
    df['gas_price_ratio_city'] = (df['gas_price_per_gallon'] / city_gas_avg.replace(0, 1.0)).astype(np.float32)
    df['gas_price_per_kw'] = (df['gas_price_per_gallon'] / (df['power_output_kw'] / 50.0).replace(0, 1.0)).astype(np.float32)
    
    # 7. Interaksi Daya dan Infrastruktur
    df['ports_total_safe'] = df['ports_total'].replace(0, 1)
    df['power_per_port'] = (df['power_output_kw'] / df['ports_total_safe']).astype(np.float32)
    df['station_total_capacity_kw'] = (df['power_output_kw'] * df['ports_total']).astype(np.float32)
    df['is_ultra_fast'] = (df['power_output_kw'] >= 150.0).astype(int)
    df['is_free_pricing'] = (df['pricing_type'].astype(str).str.lower() == 'free').astype(int)

    # 8. Fitur Domain Spesifik: Interaksi Lokasi x Jam Diurnal
    df['is_workplace_peak'] = ((df['location_type'] == 'Workplace') & (df['is_weekend'] == 0) & (df['hour'].between(8, 17))).astype(int)
    df['is_mall_peak'] = ((df['location_type'] == 'Shopping Mall') & (df['hour'].between(12, 20))).astype(int)
    df['is_highway_peak'] = ((df['location_type'] == 'Highway Corridor') & (df['hour'].between(10, 19))).astype(int)
    df['is_residential_night'] = ((df['location_type'] == 'Residential') & ((df['hour'] >= 20) | (df['hour'] <= 6))).astype(int)
    df['freezing_highway'] = (df['is_freezing'] * df['is_highway_peak']).astype(int)
    
    # 9. Penanda Acara Lokal
    df['has_local_event'] = (df['local_event'].fillna('none').astype(str).str.lower() != 'none').astype(int)

# 10. Multi-Hot Parsing Fasilitas Sekitar (Amenities)
amenities_list = ['WiFi', 'Restroom', 'Shopping Mall', 'Park', 'Fast Food', 'Hotel', 'Convenience Store', 'Grocery Store']
for amen in amenities_list:
    col_name = 'has_' + amen.lower().replace(' ', '_')
    for df in [train_ext, test_ext]:
        df[col_name] = df['amenities_nearby'].fillna('').astype(str).str.contains(amen, case=False, regex=False).astype(int)

for df in [train_ext, test_ext]:
    df['total_amenities_count'] = df[[c for c in df.columns if c.startswith('has_') and c != 'has_local_event']].sum(axis=1)

print("[INFO] Rekayasa fitur dasar spatio-temporal, cuaca, dan ekonomi selesai.")


[INFO] Rekayasa fitur dasar spatio-temporal, cuaca, dan ekonomi selesai.


In [25]:
# Hierarchical Bayesian Smoothed Macro Target Profiles (5 Pilar Teruji)
TARGET_PROFILE_COLS = [
    'target_prof_st_hr_wk',
    'target_prof_st_hr',
    'target_prof_st',
    'target_prof_loc_hr',
    'target_prof_net_hr'
]

def compute_smoothed_target_profiles(train_source, *target_dfs, m_weight=15.0):
    global_mean = train_source['utilization_rate'].mean()

    def smooth_agg(group_keys, col_name):
        agg_df = train_source.groupby(group_keys, observed=False)['utilization_rate'].agg(['count', 'mean']).reset_index()
        agg_df[col_name] = (agg_df['count'] * agg_df['mean'] + m_weight * global_mean) / (agg_df['count'] + m_weight)
        return agg_df[group_keys + [col_name]]

    # 5 Dimensi Agregasi Target
    st_hr_wk_prof = smooth_agg(['station_id', 'hour', 'is_weekend'], 'target_prof_st_hr_wk')
    st_hr_prof = smooth_agg(['station_id', 'hour'], 'target_prof_st_hr')
    st_prof = smooth_agg(['station_id'], 'target_prof_st')
    loc_hr_prof = smooth_agg(['location_type', 'hour'], 'target_prof_loc_hr')
    net_hr_prof = smooth_agg(['network', 'hour'], 'target_prof_net_hr')

    def merge_profiles(df):
        out = df.copy()
        existing = [c for c in TARGET_PROFILE_COLS if c in out.columns]
        if len(existing) > 0:
            out = out.drop(columns=existing)

        out = out.merge(st_hr_wk_prof, on=['station_id', 'hour', 'is_weekend'], how='left')
        out = out.merge(st_hr_prof, on=['station_id', 'hour'], how='left')
        out = out.merge(st_prof, on=['station_id'], how='left')
        out = out.merge(loc_hr_prof, on=['location_type', 'hour'], how='left')
        out = out.merge(net_hr_prof, on=['network', 'hour'], how='left')

        # Fallback Hierarchy Presisi
        out['target_prof_st_hr_wk'] = out['target_prof_st_hr_wk'].fillna(out['target_prof_st_hr']).fillna(out['target_prof_st']).fillna(global_mean)
        out['target_prof_st_hr'] = out['target_prof_st_hr'].fillna(out['target_prof_st']).fillna(global_mean)
        out['target_prof_st'] = out['target_prof_st'].fillna(global_mean)
        out['target_prof_loc_hr'] = out['target_prof_loc_hr'].fillna(global_mean)
        out['target_prof_net_hr'] = out['target_prof_net_hr'].fillna(global_mean)
        return out

    transformed = [merge_profiles(train_source)]
    for target_df in target_dfs:
        transformed.append(merge_profiles(target_df))
    return transformed if len(transformed) > 1 else transformed[0]

train_ext, test_ext = compute_smoothed_target_profiles(train_ext, test_ext)
print("[INFO] 5 Pilar Bayesian Smoothed Macro Target Profiles berhasil digabungkan.")


[INFO] 5 Pilar Bayesian Smoothed Macro Target Profiles berhasil digabungkan.


In [26]:
# Seleksi Fitur dan Penyiapan Tipe Kategori
# Menghapus kolom identitas murni dan target
drop_cols = ['id', 'timestamp', 'datetime', 'station_name', 'amenities_nearby', 'utilization_rate', 'ports_total_safe']
model_features = [c for c in train_ext.columns if c not in drop_cols]

cat_features = ['station_id', 'network', 'city', 'state', 'location_type', 'charger_type', 'pricing_type', 'weather_condition', 'local_event']
for c in cat_features:
    train_ext[c] = train_ext[c].fillna('missing').astype('category')
    test_ext[c] = test_ext[c].fillna('missing').astype('category')

X_train_full = train_ext[model_features]
y_train_full = train_ext['utilization_rate'].values
X_test_full = test_ext[model_features]

print(f"[INFO] Total Fitur Pelatihan Final Run 6: {len(model_features)}")
print(f"[INFO] Matriks Fitur Train Penuh: {X_train_full.shape}")
print(f"[INFO] Matriks Fitur Test Penuh : {X_test_full.shape}")


[INFO] Total Fitur Pelatihan Final Run 6: 59
[INFO] Matriks Fitur Train Penuh: (1048575, 59)
[INFO] Matriks Fitur Test Penuh : (263550, 59)


# 4. Modeling (CRISP-DM Fase 4)

### 4.1 Arsitektur Calibrated Multi-Seed Tri-Model Stacking
Untuk menghindari jebakan overfitting dari pohon yang berlebihan pada data tabular berukuran 1 juta baris, arsitektur pemodelan Run 6 Terkalibrasi menerapkan:
1. **Kalibrasi Titik Konvergensi Optimal (*Early Stopping Precision*)**:
   - LightGBM, CatBoost GPU, dan XGBoost GPU masing-masing dievaluasi menggunakan partisi validasi out-of-time dengan batas toleransi 40 ronde (*early stopping rounds = 40*).
   - Jumlah iterasi optimal (median best iterations) disimpan sebagai batas pemangkas resmi saat melatih model pada 100% data penuh.
2. **Peredaman Variansi Multi-Seed Berdaya Kuat (*Multi-Seed Variance Reduction*)**:
   - Model dilatih ulang pada 100% data latih menggunakan 3 random seeds independen (Seed 42, 100, 2024).
3. **Non-Negative Stacking Meta-Learner**:
   - Prediksi ketiga model heterogen dipadukan menggunakan regresi teratur `Ridge(alpha=10.0, positive=True, fit_intercept=False)` untuk menghasilkan bobot ensemble yang optimal dan objektif.

In [27]:
# Pembagian Partisi Validasi Out-of-Time & Evaluasi Baseline Model
train_sorted = train_ext.sort_values('datetime').reset_index(drop=True)
val_cutoff = train_sorted['datetime'].max() - pd.Timedelta(days=7)

tr_idx = train_sorted['datetime'] <= val_cutoff
va_idx = train_sorted['datetime'] > val_cutoff

tr_data = train_sorted.loc[tr_idx].copy()
va_data = train_sorted.loc[va_idx].copy()

# Hitung target profile murni dari data latih validasi (Zero Leakage)
tr_data, va_data = compute_smoothed_target_profiles(tr_data, va_data)

X_tr = tr_data[model_features].copy()
y_tr = tr_data['utilization_rate'].values
X_va = va_data[model_features].copy()
y_va = va_data['utilization_rate'].values

for c in cat_features:
    X_tr[c] = X_tr[c].astype('category')
    X_va[c] = X_va[c].astype('category')

print(f"[INFO] Partisi Validasi Out-of-Time: Latih = {len(X_tr):,} baris | Validasi = {len(X_va):,} baris")

# Baseline Ridge Regression
num_features = [c for c in model_features if c not in cat_features]
ridge_baseline = Ridge(alpha=1.0)
ridge_baseline.fit(X_tr[num_features].fillna(0), y_tr)
preds_base = np.clip(ridge_baseline.predict(X_va[num_features].fillna(0)), 0.02, 0.98)
print(f"Baseline Ridge Model -> RMSE: {root_mean_squared_error(y_va, preds_base):.5f} | R2: {r2_score(y_va, preds_base):.5f}")


[INFO] Partisi Validasi Out-of-Time: Latih = 998,250 baris | Validasi = 50,325 baris
Baseline Ridge Model -> RMSE: 0.07926 | R2: 0.93724


In [28]:
# Model 1: Multi-Seed LightGBM Regressor dengan Early Stopping Presisi
print("=== [MODEL 1: Multi-Seed LightGBM Regressor (Calibrated Early Stopping)] ===")

SEEDS = [42, 100, 2024]

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'max_depth': 10,
    'learning_rate': 0.05,
    'n_estimators': 1500,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'verbose': -1
}

# 1. Penentuan Iterasi Optimal pada Validasi
lgb_val_preds_list = []
best_iters_lgb = []

for s in SEEDS:
    m = lgb.LGBMRegressor(**{**lgb_params, 'random_state': s})
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
    )
    best_iters_lgb.append(m.best_iteration_)
    lgb_val_preds_list.append(np.clip(m.predict(X_va), 0.02, 0.98))

val_pred_lgb = np.mean(lgb_val_preds_list, axis=0)
optimal_lgb_iter = int(np.median(best_iters_lgb))
print(f"LightGBM Validation RMSE : {root_mean_squared_error(y_va, val_pred_lgb):.5f}")
print(f"LightGBM Iterasi Terbaik : {optimal_lgb_iter} pohon")

# 2. Pelatihan Multi-Seed pada 100% Data Latih Penuh
final_lgb_params = {k: v for k, v in lgb_params.items() if k != 'n_estimators'}
lgb_test_preds_list = []

t0 = time.time()
for s in SEEDS:
    print(f"Melatih LightGBM Seed {s} pada 100% data ({max(100, optimal_lgb_iter)} pohon)...")
    m = lgb.LGBMRegressor(**final_lgb_params, n_estimators=max(100, optimal_lgb_iter), random_state=s)
    m.fit(X_train_full, y_train_full)
    lgb_test_preds_list.append(np.clip(m.predict(X_test_full), 0.02, 0.98))

pred_lgb_ensemble = np.mean(lgb_test_preds_list, axis=0)
print(f"[INFO] Multi-Seed LightGBM selesai dalam {time.time()-t0:.1f} detik.\n")


=== [MODEL 1: Multi-Seed LightGBM Regressor (Calibrated Early Stopping)] ===
LightGBM Validation RMSE : 0.06839
LightGBM Iterasi Terbaik : 128 pohon
Melatih LightGBM Seed 42 pada 100% data (128 pohon)...
Melatih LightGBM Seed 100 pada 100% data (128 pohon)...
Melatih LightGBM Seed 2024 pada 100% data (128 pohon)...
[INFO] Multi-Seed LightGBM selesai dalam 52.3 detik.



In [29]:
# Model 2: Multi-Seed CatBoost Regressor GPU dengan Early Stopping Presisi
print("=== [MODEL 2: Multi-Seed CatBoost Regressor GPU (Calibrated Early Stopping)] ===")

cb_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 1500,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'verbose': 0
}
if gpu_available:
    cb_params['task_type'] = 'GPU'
    print("[INFO] CatBoost GPU diaktifkan.")
else:
    cb_params['thread_count'] = -1

X_tr_cb = X_tr.copy()
X_va_cb = X_va.copy()
for cat in cat_features:
    X_tr_cb[cat] = X_tr_cb[cat].astype(str)
    X_va_cb[cat] = X_va_cb[cat].astype(str)

# 1. Penentuan Iterasi Optimal pada Validasi
cb_val_preds_list = []
best_iters_cb = []

for s in SEEDS:
    m = cb.CatBoostRegressor(**{**cb_params, 'random_seed': s})
    m.fit(
        X_tr_cb, y_tr,
        eval_set=(X_va_cb, y_va),
        cat_features=cat_features,
        early_stopping_rounds=40
    )
    best_iters_cb.append(m.get_best_iteration())
    cb_val_preds_list.append(np.clip(m.predict(X_va_cb), 0.02, 0.98))

val_pred_cb = np.mean(cb_val_preds_list, axis=0)
optimal_cb_iter = int(np.median(best_iters_cb))
print(f"CatBoost Validation RMSE : {root_mean_squared_error(y_va, val_pred_cb):.5f}")
print(f"CatBoost Iterasi Terbaik : {optimal_cb_iter} pohon")

# 2. Pelatihan Multi-Seed pada 100% Data Latih Penuh
X_full_cb = X_train_full.copy()
X_test_cb = X_test_full.copy()
for cat in cat_features:
    X_full_cb[cat] = X_full_cb[cat].astype(str)
    X_test_cb[cat] = X_test_cb[cat].astype(str)

final_cb_params = {k: v for k, v in cb_params.items() if k != 'iterations'}
cb_test_preds_list = []

t0 = time.time()
for s in SEEDS:
    print(f"Melatih CatBoost Seed {s} pada 100% data ({max(100, optimal_cb_iter)} pohon)...")
    m = cb.CatBoostRegressor(**final_cb_params, iterations=max(100, optimal_cb_iter), random_seed=s)
    m.fit(X_full_cb, y_train_full, cat_features=cat_features)
    cb_test_preds_list.append(np.clip(m.predict(X_test_cb), 0.02, 0.98))

pred_cb_ensemble = np.mean(cb_test_preds_list, axis=0)
print(f"[INFO] Multi-Seed CatBoost selesai dalam {time.time()-t0:.1f} detik.\n")


=== [MODEL 2: Multi-Seed CatBoost Regressor GPU (Calibrated Early Stopping)] ===
[INFO] CatBoost GPU diaktifkan.
CatBoost Validation RMSE : 0.06821
CatBoost Iterasi Terbaik : 290 pohon
Melatih CatBoost Seed 42 pada 100% data (290 pohon)...
Melatih CatBoost Seed 100 pada 100% data (290 pohon)...
Melatih CatBoost Seed 2024 pada 100% data (290 pohon)...
[INFO] Multi-Seed CatBoost selesai dalam 117.2 detik.



In [30]:
# Model 3: Multi-Seed XGBoost Regressor GPU dengan Early Stopping Presisi
print("=== [MODEL 3: Multi-Seed XGBoost Regressor GPU (Calibrated Early Stopping)] ===")

xgb_train = X_tr.copy()
xgb_val = X_va.copy()
for c in cat_features:
    xgb_train[c] = xgb_train[c].cat.codes
    xgb_val[c] = xgb_val[c].cat.codes

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'n_estimators': 1500,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 40
}
if gpu_available:
    xgb_params['device'] = 'cuda'
    print("[INFO] XGBoost GPU diaktifkan.")

# 1. Penentuan Iterasi Optimal pada Validasi
xgb_val_preds_list = []
best_iters_xgb = []

for s in SEEDS:
    m = xgb.XGBRegressor(**{**xgb_params, 'random_state': s})
    m.fit(
        xgb_train, y_tr,
        eval_set=[(xgb_val, y_va)],
        verbose=False
    )
    best_iters_xgb.append(m.best_iteration)
    xgb_val_preds_list.append(np.clip(m.predict(xgb_val), 0.02, 0.98))

val_pred_xgb = np.mean(xgb_val_preds_list, axis=0)
optimal_xgb_iter = int(np.median(best_iters_xgb))
print(f"XGBoost Validation RMSE  : {root_mean_squared_error(y_va, val_pred_xgb):.5f}")
print(f"XGBoost Iterasi Terbaik  : {optimal_xgb_iter} pohon")

# 2. Pelatihan Multi-Seed pada 100% Data Latih Penuh
xgb_full_tr = X_train_full.copy()
xgb_full_te = X_test_full.copy()
for c in cat_features:
    xgb_full_tr[c] = xgb_full_tr[c].cat.codes
    xgb_full_te[c] = xgb_full_te[c].cat.codes

final_xgb_params = {k: v for k, v in xgb_params.items() if k not in ['n_estimators', 'early_stopping_rounds']}
xgb_test_preds_list = []

t0 = time.time()
for s in SEEDS:
    print(f"Melatih XGBoost Seed {s} pada 100% data ({max(100, optimal_xgb_iter)} pohon)...")
    m = xgb.XGBRegressor(**final_xgb_params, n_estimators=max(100, optimal_xgb_iter), random_state=s)
    m.fit(xgb_full_tr, y_train_full, verbose=False)
    xgb_test_preds_list.append(np.clip(m.predict(xgb_full_te), 0.02, 0.98))

pred_xgb_ensemble = np.mean(xgb_test_preds_list, axis=0)
print(f"[INFO] Multi-Seed XGBoost selesai dalam {time.time()-t0:.1f} detik.\n")


=== [MODEL 3: Multi-Seed XGBoost Regressor GPU (Calibrated Early Stopping)] ===
[INFO] XGBoost GPU diaktifkan.
XGBoost Validation RMSE  : 0.06831
XGBoost Iterasi Terbaik  : 143 pohon
Melatih XGBoost Seed 42 pada 100% data (143 pohon)...
Melatih XGBoost Seed 100 pada 100% data (143 pohon)...
Melatih XGBoost Seed 2024 pada 100% data (143 pohon)...
[INFO] Multi-Seed XGBoost selesai dalam 15.7 detik.



In [31]:
# Ensembling: Non-Negative Stacking Meta-Learner (Ridge Regularized)
print("=== [OPTIMASI STACKING META-LEARNER TRI-MODEL RUN 6 TERKALIBRASI] ===")

S_va = np.column_stack([val_pred_lgb, val_pred_cb, val_pred_xgb])

meta_learner = Ridge(alpha=10.0, positive=True, fit_intercept=False)
meta_learner.fit(S_va, y_va)

stacking_val_preds = np.clip(meta_learner.predict(S_va), 0.02, 0.98)
rmse_stacking = root_mean_squared_error(y_va, stacking_val_preds)

print("Bobot Meta-Learner Terkalibrasi:")
print(f"  Bobot LightGBM Multi-Seed : {meta_learner.coef_[0]:.4f}")
print(f"  Bobot CatBoost Multi-Seed : {meta_learner.coef_[1]:.4f}")
print(f"  Bobot XGBoost Multi-Seed  : {meta_learner.coef_[2]:.4f}")
print(f"Validasi RMSE Hasil Stacking Terpadu: {rmse_stacking:.5f}")

# Penggabungan Prediksi Data Uji Multi-Seed Tri-Model
S_test = np.column_stack([pred_lgb_ensemble, pred_cb_ensemble, pred_xgb_ensemble])
integrated_ensemble_pred = np.clip(meta_learner.predict(S_test), 0.02, 0.98)
print(f"Statistik Prediksi Final Tri-Model Run 6: Min={integrated_ensemble_pred.min():.4f}, Max={integrated_ensemble_pred.max():.4f}, Mean={integrated_ensemble_pred.mean():.4f}")


=== [OPTIMASI STACKING META-LEARNER TRI-MODEL RUN 6 TERKALIBRASI] ===
Bobot Meta-Learner Terkalibrasi:
  Bobot LightGBM Multi-Seed : 0.3251
  Bobot CatBoost Multi-Seed : 0.3451
  Bobot XGBoost Multi-Seed  : 0.3304
Validasi RMSE Hasil Stacking Terpadu: 0.06821
Statistik Prediksi Final Tri-Model Run 6: Min=0.0201, Max=0.9800, Mean=0.4499


# 5. Evaluation (CRISP-DM Fase 5)

### 5.1 Karakterisasi Evaluasi Komparatif dan Pasca-Pemrosesan Batas Fisik
Evaluasi perbandingan metrik galat dan kuantisasi sensor dilakukan untuk memastikan prediksi memenuhi batasan fisik stasiun pengisian daya EV di dunia nyata.

In [32]:
# Pasca-Pemrosesan Batas Fisik Operasional dan Presisi Kuantisasi Sensor
# 1. Pemotongan Batas Fisik Operasional [0.02, 0.98] Sesuai Karakteristik Stasiun Nyata
final_predictions = np.clip(integrated_ensemble_pred, 0.02, 0.98)

# 2. Pembulatan Tiga Angka Desimal Sesuai Spesifikasi Resolusi Sensor Ground Truth
final_predictions = np.round(final_predictions, 3)

print(f"[INFO] Prediksi Final Multi-Seed Tri-Model Run 6 Selesai. Total baris: {len(final_predictions):,}")
print(f"[INFO] Statistik Prediksi Final:")
print(f"  Nilai Minimum  : {final_predictions.min():.3f}")
print(f"  Nilai Maksimum : {final_predictions.max():.3f}")
print(f"  Nilai Rata-rata: {final_predictions.mean():.3f}")
print(f"  Standar Deviasi: {final_predictions.std():.3f}")


[INFO] Prediksi Final Multi-Seed Tri-Model Run 6 Selesai. Total baris: 263,550
[INFO] Statistik Prediksi Final:
  Nilai Minimum  : 0.020
  Nilai Maksimum : 0.980
  Nilai Rata-rata: 0.450
  Standar Deviasi: 0.307


# 6. Deployment (CRISP-DM Fase 6)

### 6.1 Pembangkitan Berkas Submission Resmi
Berkas submission resmi dibentuk dengan skema yang ditetapkan oleh panitia:
* Kolom: `id` dan `utilization_rate`
* Format Penamaan: `MAKAN ITU PENTING_Submission.csv`
* Dimensi: Tepat 263.550 baris sesuai data pengujian.

In [33]:
# Pembangkitan dan Verifikasi Berkas Submission Resmi
submission = pd.DataFrame({
    'id': test_df['id'],
    'utilization_rate': final_predictions
})

# Verifikasi Integritas
assert len(submission) == len(test_df), f"Dimensi tidak cocok: {len(submission)} vs {len(test_df)}"
assert not submission['utilization_rate'].isnull().any(), "Terdapat nilai NaN pada berkas submission!"
assert (submission['utilization_rate'] >= 0.02).all() and (submission['utilization_rate'] <= 0.98).all(), "Nilai melampaui batas fisik!"

SUBMISSION_OUT = 'MAKAN ITU PENTING_Submission.csv'
submission.to_csv(SUBMISSION_OUT, index=False)

# Simpan salinan ke folder Submission/Run 6 jika direktori tersedia
run6_dir = './Submission/Run 6'
if os.path.exists(run6_dir):
    submission.to_csv(os.path.join(run6_dir, 'MAKAN ITU PENTING_Submission.csv'), index=False)
    print(f"[INFO] Salinan berkas submission berhasil disimpan di {run6_dir}.")

print(f"=== BERKAS SUBMISSION RUN 6 TERKALIBRASI BERHASIL DIBENTUK ===")
print(f"Nama Berkas : {SUBMISSION_OUT}")
print(f"Dimensi     : {submission.shape[0]:,} baris x {submission.shape[1]} kolom")
print(f"Sampel 5 Baris Pertama:\n{submission.head()}")


=== BERKAS SUBMISSION RUN 6 TERKALIBRASI BERHASIL DIBENTUK ===
Nama Berkas : MAKAN ITU PENTING_Submission.csv
Dimensi     : 263,550 baris x 2 kolom
Sampel 5 Baris Pertama:
           id  utilization_rate
0  TST_0U33RJ             0.963
1  TST_V0LFA4             0.802
2  TST_J07LFI             0.614
3  TST_K4KBB3             0.932
4  TST_OB0S2M             0.567


### 6.2 Rekomendasi Strategis Berkelanjutan (SDG 7 & SDG 9)

Berdasarkan wawasan model prediktif Multi-Seed Tri-Model Stacking Run 6 Terkalibrasi, dirumuskan tiga rekomendasi strategis:
1. **Dinamisasi Tarif Berbasis Slot Waktu Granular (Time-of-Use Granular Pricing)**:
   Variasi beban yang signifikan antara menit :00 dan :30 pada jam sibuk membuktikan perlunya tarif insentif dinamis per interval 30 menit untuk meratakan kurva beban harian (peak shaving).
2. **Kompensasi Termal Gardu dan Koridor Musim Dingin**:
   Lonjakan utilisasi pada cuaca beku menuntut operator untuk memperkuat daya cadangan baterai stasioner lokal (Battery Energy Storage System / BESS) pada stasiun koridor bebas hambatan selama musim dingin.
3. **Penyelarasan Kapasitas Port dengan Fasilitas Penunjang**:
   Stasiun pengisian di pusat perbelanjaan dan pusat transit komersial memiliki retensi pengisian lebih lama, sehingga penambahan port pengisian daya berdaya menengah (50 kW - 150 kW) lebih efektif dibandingkan hanya menambah port lambat.